# Power Series, Taylor Expansions, and Applications — An Interactive Laboratory

**Companion notebook for Chapter 4 of _Advanced Calculus of One Variable with Python and AI_.**

A power series

$$
\sum_{n=0}^{\infty}a_n(x-x_0)^n
$$

combines convergence theory, approximation, probability, and numerical algorithms. Its radius controls the interior, Taylor's theorem certifies finite approximations, and generating functions turn distributions into analytic objects.

### Learning goals

1. Determine a radius and test both endpoints separately.
2. Explore compact uniform convergence and term-by-term operations.
3. Visualize Abel's boundary theorem.
4. Construct Taylor polynomials with rigorous remainder bounds.
5. Distinguish smoothness from analyticity.
6. Use series to evaluate constants and integrals.
7. Work with probability generating functions and branching processes.
8. Compare bisection, Newton iteration, and the trapezoidal rule.
9. Relate infinite products to logarithmic series.

Run the cells from top to bottom. Every laboratory has a blue action button.

## Cell 1 — Imports, display style, and reusable helpers

The Cauchy--Hadamard formula

$$
\frac1R=\limsup_{n\to\infty}|a_n|^{1/n}
$$

and the error estimates used later require stable numerical evaluation. This setup cell loads NumPy, SciPy, Matplotlib, and the widget system. Missing packages are installed once in the active notebook environment.

In [ ]:
import sys
import subprocess
import importlib


# Install only packages missing from the current notebook runtime.
required_packages = {
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "scipy": "scipy",
    "ipywidgets": "ipywidgets",
    "IPython": "ipython",
}

for module_name, package_name in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        print(f"Installing missing package: {package_name}")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package_name]
        )

import math
import random
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, Markdown, HTML, clear_output
from scipy.integrate import quad
from scipy.optimize import brentq
from scipy.special import gammaln, spence


plt.style.use("seaborn-v0_8-whitegrid")
COLORS = {
    "blue": "#2563eb",
    "orange": "#ea580c",
    "green": "#15803d",
    "red": "#dc2626",
    "purple": "#7e22ce",
    "gray": "#475569",
}


def style_button(button, icon="play"):
    """Apply a consistent visual style to every action button."""
    button.button_style = "primary"
    button.icon = icon
    button.layout = widgets.Layout(width="190px")
    return button


def show_note(text, color="#eff6ff", border=None):
    """Display a concise theorem, interpretation, or warning note."""
    border_color = border or COLORS["blue"]
    display(
        HTML(
            f"<div style='padding:10px 14px;border-left:4px solid {border_color};"
            f"background:{color};margin:8px 0'>{text}</div>"
        )
    )


display(
    HTML(
        "<div style='padding:12px 16px;background:#ecfdf5;border-left:5px solid #15803d'>"
        "<b>Setup complete.</b> The power-series laboratories are ready.</div>"
    )
)

## Cell 2 — Radius of convergence and independent endpoint tests

The radius gives three regions:

$$
|x-x_0|<R:\text{ absolute convergence},\qquad
|x-x_0|>R:\text{ divergence},
$$

while $|x-x_0|=R$ requires a separate numerical-series test at each endpoint. Select a coefficient family and compare the root estimates $|a_n|^{1/n}$ with the terms and partial sums at a chosen $x$.

In [ ]:
radius_family = widgets.Dropdown(
    options=[
        ("a_n=1/(n+1): left endpoint only", "harmonic"),
        ("a_n=1/(n+1)^2: both endpoints", "square"),
        ("a_n=(-1)^n/(n+1): right endpoint only", "signed"),
        ("a_n=2^(-n): neither endpoint", "geometric"),
        ("a_n=1/n!: infinite radius", "factorial_inverse"),
        ("a_n=n!: zero radius", "factorial"),
    ],
    value="harmonic",
    description="Coefficients:",
    layout=widgets.Layout(width="540px"),
)
radius_x = widgets.FloatSlider(
    value=0.80, min=-1.30, max=1.30, step=0.05, description="x:",
    continuous_update=False, layout=widgets.Layout(width="540px")
)
radius_N = widgets.IntSlider(
    value=50, min=5, max=120, step=1, description="N:",
    continuous_update=False, layout=widgets.Layout(width="540px")
)
radius_button = style_button(widgets.Button(description="Explore convergence"), "bullseye")
radius_output = widgets.Output()


def power_coefficients(family, N):
    """Return coefficients a_0,...,a_N for the selected family."""
    n = np.arange(N + 1, dtype=float)
    if family == "harmonic":
        return 1.0 / (n + 1.0)
    if family == "square":
        return 1.0 / (n + 1.0) ** 2
    if family == "signed":
        return (-1.0) ** n.astype(int) / (n + 1.0)
    if family == "geometric":
        return 2.0 ** (-n)
    if family == "factorial_inverse":
        return np.exp(-gammaln(n + 1.0))
    return np.exp(gammaln(n + 1.0))


def radius_metadata(family):
    data = {
        "harmonic": (1.0, "interval $[-1,1)$; left conditional, right divergent"),
        "square": (1.0, "interval $[-1,1]$; both endpoints absolutely convergent"),
        "signed": (1.0, "interval $(-1,1]$; right conditional, left divergent"),
        "geometric": (2.0, "interval $(-2,2)$; terms fail to vanish at both endpoints"),
        "factorial_inverse": (math.inf, "converges for every real $x$"),
        "factorial": (0.0, "converges only at the centre $x=0$"),
    }
    return data[family]


def run_radius_lab(_=None):
    with radius_output:
        clear_output(wait=True)
        family, x_value, N = radius_family.value, radius_x.value, radius_N.value
        coefficients = power_coefficients(family, N)
        n = np.arange(N + 1)
        terms = coefficients * x_value**n
        sums = np.cumsum(terms)
        root_n = np.arange(1, N + 1)
        roots = np.exp(np.log(np.abs(coefficients[1:])) / root_n)
        R, classification = radius_metadata(family)

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
        axes[0].plot(root_n, roots, color=COLORS["purple"], lw=2)
        if math.isfinite(R) and R > 0:
            axes[0].axhline(1.0 / R, color=COLORS["red"], ls="--", label="$1/R$")
        axes[0].set_title(r"Coefficient roots $|a_n|^{1/n}$")
        axes[0].set_xlabel("$n$")
        axes[0].legend() if math.isfinite(R) and R > 0 else None
        axes[1].plot(n, sums, color=COLORS["blue"], lw=2)
        axes[1].set_title(rf"Partial sums at $x={x_value:.2f}$")
        axes[1].set_xlabel("$N$")
        plt.tight_layout()
        plt.show()

        radius_text = "$+\\infty$" if math.isinf(R) else f"{R:g}"
        print(f"Theoretical radius R = {radius_text}")
        print(f"Last term             = {terms[-1]:.10g}")
        print(f"Partial sum S_N       = {sums[-1]:.10g}")
        show_note(classification)


def configure_radius(change=None):
    family = radius_family.value
    if family == "geometric":
        radius_x.min, radius_x.max, radius_x.step = -2.4, 2.4, 0.1
    elif family == "factorial_inverse":
        radius_x.min, radius_x.max, radius_x.step = -5.0, 5.0, 0.1
    elif family == "factorial":
        radius_x.min, radius_x.max, radius_x.step = -0.5, 0.5, 0.02
    else:
        radius_x.min, radius_x.max, radius_x.step = -1.3, 1.3, 0.05


radius_family.observe(configure_radius, names="value")
radius_button.on_click(run_radius_lab)
configure_radius()
display(widgets.VBox([radius_family, radius_x, radius_N, radius_button, radius_output]))
run_radius_lab()

## Cell 3 — Compact uniform convergence and term-by-term operations

Inside the radius, every smaller closed interval enjoys uniform convergence. Starting from

$$
\sum_{n=0}^{\infty}x^n=\frac1{1-x},\qquad |x|<1,
$$

term-by-term differentiation and integration give

$$
\sum_{n=1}^{\infty}nx^{n-1}=\frac1{(1-x)^2},
\qquad
\sum_{n=0}^{\infty}\frac{x^{n+1}}{n+1}=-\log(1-x).
$$

All three series have radius $1$. The uniform remainder of the original geometric series on $[-r,r]$ is at most $r^{N+1}/(1-r)$.

In [ ]:
operations_r = widgets.FloatSlider(
    value=0.75, min=0.10, max=0.98, step=0.01, description="r:",
    continuous_update=False, layout=widgets.Layout(width="480px")
)
operations_N = widgets.IntSlider(
    value=15, min=1, max=100, step=1, description="N:",
    continuous_update=False, layout=widgets.Layout(width="480px")
)
operations_button = style_button(widgets.Button(description="Operate on series"), "project-diagram")
operations_output = widgets.Output()


def geometric_operation_data(r, N, grid_size=1601):
    x = np.linspace(-r, r, grid_size)
    n = np.arange(N + 1)
    partial = np.sum(x[:, None] ** n[None, :], axis=1)
    exact = 1.0 / (1.0 - x)
    if N >= 1:
        k = np.arange(1, N + 1)
        derivative_partial = np.sum(k[None, :] * x[:, None] ** (k[None, :] - 1), axis=1)
    else:
        derivative_partial = np.zeros_like(x)
    derivative_exact = 1.0 / (1.0 - x) ** 2
    integral_partial = np.sum(x[:, None] ** (n[None, :] + 1) / (n[None, :] + 1), axis=1)
    integral_exact = -np.log(1.0 - x)
    return x, partial, exact, derivative_partial, derivative_exact, integral_partial, integral_exact


def run_operations_lab(_=None):
    with operations_output:
        clear_output(wait=True)
        r, N = operations_r.value, operations_N.value
        data = geometric_operation_data(r, N)
        x, partial, exact, d_partial, d_exact, i_partial, i_exact = data
        bound = r ** (N + 1) / (1.0 - r)

        fig, axes = plt.subplots(1, 3, figsize=(14, 4.0))
        pairs = [
            (partial, exact, "Original series"),
            (d_partial, d_exact, "Differentiated series"),
            (i_partial, i_exact, "Integrated series"),
        ]
        for ax, (approx, truth, title) in zip(axes, pairs):
            ax.plot(x, truth, color=COLORS["red"], ls="--", label="exact")
            ax.plot(x, approx, color=COLORS["blue"], lw=1.8, label="$N$th approximation")
            ax.set_title(title)
            ax.legend()
        plt.tight_layout()
        plt.show()

        print(f"Geometric uniform error       = {np.max(np.abs(partial-exact)):.6e}")
        print(f"Geometric rigorous bound      = {bound:.6e}")
        print(f"Derivative displayed sup error= {np.max(np.abs(d_partial-d_exact)):.6e}")
        print(f"Integral displayed sup error  = {np.max(np.abs(i_partial-i_exact)):.6e}")
        show_note("Term-by-term differentiation and integration are justified on every closed interval strictly inside the radius.")


operations_button.on_click(run_operations_lab)
display(widgets.VBox([operations_r, operations_N, operations_button, operations_output]))

## Cell 4 — Abel's theorem at the boundary

If $\sum a_n=A$, Abel's theorem gives

$$
\lim_{r\uparrow1}\sum_{n=0}^{\infty}a_nr^n=A.
$$

It is a one-sided continuity theorem from inside the disk of convergence. For nonnegative coefficients, the finite or infinite boundary behaviour is equivalent to the limit of the radial sums.

In [ ]:
abel_family = widgets.Dropdown(
    options=[
        ("Alternating harmonic: boundary value log(2)", "alternating"),
        ("Reciprocal squares: boundary value pi^2/6", "squares"),
        ("Positive harmonic: radial sum diverges", "harmonic"),
    ],
    value="alternating",
    description="Coefficients:",
    layout=widgets.Layout(width="520px"),
)
abel_r = widgets.FloatSlider(
    value=0.90, min=0.0, max=0.999, step=0.001, description="r:",
    continuous_update=False, readout_format=".3f", layout=widgets.Layout(width="520px")
)
abel_N = widgets.IntSlider(
    value=300, min=10, max=5000, step=10, description="N:",
    continuous_update=False, layout=widgets.Layout(width="520px")
)
abel_button = style_button(widgets.Button(description="Approach boundary"), "arrow-right")
abel_output = widgets.Output()


def abel_exact(family, r):
    if family == "alternating":
        return math.log1p(r) / r if r > 0 else 1.0, math.log(2.0)
    if family == "squares":
        # scipy.special.spence(1-r) equals Li_2(r).
        return (spence(1.0 - r) / r if r > 0 else 1.0), math.pi**2 / 6.0
    return (-math.log1p(-r) / r if r > 0 else 1.0), math.inf


def abel_coefficients(family, N):
    n = np.arange(N + 1, dtype=float)
    if family == "alternating":
        return (-1.0) ** n.astype(int) / (n + 1.0)
    if family == "squares":
        return 1.0 / (n + 1.0) ** 2
    return 1.0 / (n + 1.0)


def run_abel_boundary(_=None):
    with abel_output:
        clear_output(wait=True)
        family, r, N = abel_family.value, abel_r.value, abel_N.value
        n = np.arange(N + 1)
        coefficients = abel_coefficients(family, N)
        partial = float(np.sum(coefficients * r**n))
        exact, boundary = abel_exact(family, r)
        r_grid = np.linspace(0.0, 0.995, 300)
        curve = np.array([abel_exact(family, value)[0] for value in r_grid])

        fig, ax = plt.subplots(figsize=(10.4, 4.2))
        ax.plot(r_grid, curve, color=COLORS["blue"], lw=2, label=r"$\sum a_nr^n$")
        ax.scatter([r], [exact], color=COLORS["red"], s=60, zorder=3, label="selected r")
        if math.isfinite(boundary):
            ax.axhline(boundary, color=COLORS["green"], ls="--", label="endpoint sum")
        ax.set_xlabel("$r$")
        ax.set_title("Radial approach to the boundary")
        ax.legend()
        plt.show()

        print(f"Truncated radial sum through n={N} = {partial:.12f}")
        print(f"Exact radial value                 = {exact:.12f}")
        print(f"Truncation error                   = {abs(partial-exact):.3e}")
        if math.isfinite(boundary):
            print(f"Boundary coefficient sum           = {boundary:.12f}")
            show_note("The radial values approach the convergent endpoint series from within the radius.")
        else:
            show_note(r"The nonnegative coefficient series diverges, and its radial sums grow without bound as $r\uparrow1$.", "#fff7ed", COLORS["orange"])


abel_button.on_click(run_abel_boundary)
display(widgets.VBox([abel_family, abel_r, abel_N, abel_button, abel_output]))

## Cell 5 — Taylor polynomials and rigorous Lagrange bounds

Taylor's theorem writes

$$
f(x)=T_n(f;0)(x)+R_n(x),
\qquad
|R_n(x)|\le\frac{M|x|^{n+1}}{(n+1)!},
$$

when $|f^{(n+1)}(t)|\le M$ between $0$ and $x$. The theorem provides a certified error, not merely an observed difference.

In [ ]:
taylor_family = widgets.Dropdown(
    options=[
        ("Exponential e^x", "exp"),
        ("Sine sin(x)", "sin"),
        ("Cosine cos(x)", "cos"),
        ("Logarithm log(1+x)", "log"),
        ("Arctangent arctan(x)", "arctan"),
    ],
    value="exp",
    description="Function:",
    layout=widgets.Layout(width="490px"),
)
taylor_x = widgets.FloatSlider(
    value=1.0, min=-2.0, max=2.0, step=0.05, description="x:",
    continuous_update=False, layout=widgets.Layout(width="490px")
)
taylor_degree = widgets.IntSlider(
    value=6, min=0, max=30, step=1, description="Degree n:",
    continuous_update=False, layout=widgets.Layout(width="490px")
)
taylor_button = style_button(widgets.Button(description="Certify approximation"), "certificate")
taylor_output = widgets.Output()


def taylor_polynomial(family, x, degree):
    x = np.asarray(x, dtype=float)
    result = np.zeros_like(x)
    if family == "exp":
        term = np.ones_like(x)
        result += term
        for k in range(1, degree + 1):
            term = term * x / k
            result += term
    elif family == "sin":
        for k in range((degree + 1) // 2):
            power = 2 * k + 1
            if power <= degree:
                result += (-1.0) ** k * x**power / math.factorial(power)
    elif family == "cos":
        for k in range(degree // 2 + 1):
            power = 2 * k
            if power <= degree:
                result += (-1.0) ** k * x**power / math.factorial(power)
    elif family == "log":
        for k in range(1, degree + 1):
            result += (-1.0) ** (k - 1) * x**k / k
    else:
        for k in range((degree + 1) // 2):
            power = 2 * k + 1
            if power <= degree:
                result += (-1.0) ** k * x**power / power
    return result


def exact_function(family, x):
    functions = {
        "exp": np.exp,
        "sin": np.sin,
        "cos": np.cos,
        "log": np.log1p,
        "arctan": np.arctan,
    }
    return functions[family](x)


def taylor_error_bound(family, x, degree):
    distance = abs(float(x))
    if family == "exp":
        return math.exp(distance) * distance ** (degree + 1) / math.factorial(degree + 1), "Lagrange"
    if family in {"sin", "cos"}:
        return distance ** (degree + 1) / math.factorial(degree + 1), "Lagrange"
    if family == "log":
        lower = min(1.0, 1.0 + float(x))
        bound = distance ** (degree + 1) / ((degree + 1) * lower ** (degree + 1))
        return bound, "Lagrange"
    # For |x|<=1, the alternating arctangent series has first-omitted-term error.
    next_power = degree + 1 if (degree + 1) % 2 == 1 else degree + 2
    return distance**next_power / next_power, "alternating-series"


def run_taylor_lab(_=None):
    with taylor_output:
        clear_output(wait=True)
        family, x_value, degree = taylor_family.value, taylor_x.value, taylor_degree.value
        approximation = float(taylor_polynomial(family, np.array([x_value]), degree)[0])
        exact = float(exact_function(family, np.array([x_value]))[0])
        bound, bound_name = taylor_error_bound(family, x_value, degree)

        A = max(1.0, abs(x_value))
        if family in {"log", "arctan"}:
            A = min(A, 0.95)
        grid = np.linspace(-A, A, 1201)
        truth = exact_function(family, grid)
        polynomial = taylor_polynomial(family, grid, degree)

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.1))
        axes[0].plot(grid, truth, color=COLORS["red"], ls="--", label="function")
        axes[0].plot(grid, polynomial, color=COLORS["blue"], label="$T_n$")
        axes[0].scatter([x_value], [exact], color=COLORS["green"], zorder=3)
        axes[0].set_title("Function and Taylor polynomial")
        axes[0].legend()
        axes[1].semilogy(grid, np.maximum(np.abs(truth-polynomial), 1e-18), color=COLORS["purple"])
        axes[1].set_title("Absolute approximation error")
        plt.tight_layout()
        plt.show()

        print(f"T_n(x)                   = {approximation:.14g}")
        print(f"Exact f(x)               = {exact:.14g}")
        print(f"Observed absolute error  = {abs(exact-approximation):.6e}")
        print(f"{bound_name} bound = {bound:.6e}")
        if abs(exact-approximation) <= bound + 1e-12:
            show_note("The observed error lies inside the rigorous theorem bound.")
        else:
            show_note("The selected polynomial degree and theorem bound are incompatible; inspect the hypotheses.", "#fef2f2", COLORS["red"])


def configure_taylor(change=None):
    if taylor_family.value in {"log", "arctan"}:
        taylor_x.min, taylor_x.max, taylor_x.step = -0.90, 0.90, 0.05
        if abs(taylor_x.value) > 0.90:
            taylor_x.value = 0.50
    else:
        taylor_x.min, taylor_x.max, taylor_x.step = -3.0, 3.0, 0.05


taylor_family.observe(configure_taylor, names="value")
taylor_button.on_click(run_taylor_lab)
configure_taylor()
display(widgets.VBox([taylor_family, taylor_x, taylor_degree, taylor_button, taylor_output]))

## Cell 6 — Series as certified computational tools

Standard expansions can be transformed into rapidly convergent formulas. This laboratory compares:

$$
\log2=2\sum_{n=0}^{\infty}\frac{1}{(2n+1)3^{2n+1}},
$$

$$
\int_0^1e^{-t^2/2}\,dt
=\sum_{n=0}^{\infty}(-1)^n\frac{1}{2^n n!(2n+1)},
$$

and the series for $\int_0^{1/3}(1+x^3)^{-1}\,dx$. Each alternating remainder is bounded by the first omitted term.

In [ ]:
application_family = widgets.Dropdown(
    options=[
        ("Fast computation of log(2)", "log2"),
        ("Gaussian integral from 0 to 1", "gaussian"),
        ("Integral of 1/(1+x^3) from 0 to 1/3", "rational_integral"),
    ],
    value="log2",
    description="Application:",
    layout=widgets.Layout(width="540px"),
)
application_N = widgets.IntSlider(
    value=6, min=0, max=40, step=1, description="Last index N:",
    continuous_update=False, layout=widgets.Layout(width="540px")
)
application_button = style_button(widgets.Button(description="Compute with series"), "calculator")
application_output = widgets.Output()


def series_application_values(family, N):
    n = np.arange(N + 1, dtype=float)
    if family == "log2":
        terms = 2.0 / ((2.0 * n + 1.0) * 3.0 ** (2.0 * n + 1.0))
        exact = math.log(2.0)
        next_term = 2.0 / ((2.0 * (N + 1) + 1.0) * 3.0 ** (2.0 * (N + 1) + 1.0))
        # This is a positive series; bound its tail by a geometric comparison.
        bound = next_term / (1.0 - 1.0 / 9.0)
    elif family == "gaussian":
        log_magnitude = -(n * math.log(2.0) + gammaln(n + 1.0) + np.log(2.0 * n + 1.0))
        terms = (-1.0) ** n.astype(int) * np.exp(log_magnitude)
        exact = quad(lambda t: math.exp(-t*t/2.0), 0.0, 1.0)[0]
        k = N + 1
        bound = math.exp(-(k * math.log(2.0) + gammaln(k + 1.0) + math.log(2.0 * k + 1.0)))
    else:
        terms = (-1.0) ** n.astype(int) / ((3.0 * n + 1.0) * 3.0 ** (3.0 * n + 1.0))
        exact = quad(lambda t: 1.0 / (1.0 + t**3), 0.0, 1.0/3.0)[0]
        k = N + 1
        bound = 1.0 / ((3.0 * k + 1.0) * 3.0 ** (3.0 * k + 1.0))
    partials = np.cumsum(terms)
    return partials, exact, bound


def run_series_application(_=None):
    with application_output:
        clear_output(wait=True)
        family, N = application_family.value, application_N.value
        partials, exact, bound = series_application_values(family, N)
        errors = np.abs(partials - exact)

        fig, ax = plt.subplots(figsize=(10.3, 4.1))
        ax.semilogy(np.arange(N + 1), np.maximum(errors, 1e-18), marker="o", color=COLORS["purple"])
        ax.set_xlabel("last included index $N$")
        ax.set_ylabel("absolute error")
        ax.set_title("Convergence of the certified series approximation")
        plt.show()

        print(f"Series approximation = {partials[-1]:.15g}")
        print(f"Reference value       = {exact:.15g}")
        print(f"Observed error        = {errors[-1]:.6e}")
        print(f"Rigorous tail bound   = {bound:.6e}")
        show_note("The bound is derived analytically from the first omitted term or a geometric tail, not from the reference value.")


application_button.on_click(run_series_application)
display(widgets.VBox([application_family, application_N, application_button, application_output]))

## Cell 7 — Smooth does not imply analytic

Define

$$
f(x)=\begin{cases}e^{-1/x^2},&x\ne0,\\0,&x=0.\end{cases}
$$

Every derivative at $0$ equals zero, so the Taylor series at $0$ is identically zero. Nevertheless, $f(x)>0$ away from the centre. The function is $C^\infty$ but is not represented by its Taylor series near $0$.

In [ ]:
flat_A = widgets.FloatSlider(
    value=0.60, min=0.10, max=2.0, step=0.05, description="Display A:",
    continuous_update=False, layout=widgets.Layout(width="480px")
)
flat_scale = widgets.Checkbox(value=True, description="Use logarithmic vertical scale")
flat_button = style_button(widgets.Button(description="Compare with Taylor"), "not-equal")
flat_output = widgets.Output()


def flat_function(x):
    x = np.asarray(x, dtype=float)
    result = np.zeros_like(x)
    mask = x != 0.0
    result[mask] = np.exp(-1.0 / x[mask] ** 2)
    return result


def run_flat_lab(_=None):
    with flat_output:
        clear_output(wait=True)
        A = flat_A.value
        x = np.linspace(-A, A, 3001)
        values = flat_function(x)
        taylor = np.zeros_like(x)

        fig, ax = plt.subplots(figsize=(10.3, 4.2))
        if flat_scale.value:
            ax.semilogy(x, np.maximum(values, 1e-300), color=COLORS["blue"], lw=2, label="$f(x)$")
            ax.axhline(1e-300, color=COLORS["red"], ls="--", label="zero Taylor series (display floor)")
        else:
            ax.plot(x, values, color=COLORS["blue"], lw=2, label="$f(x)$")
            ax.plot(x, taylor, color=COLORS["red"], ls="--", label="Taylor series")
        ax.set_title("A flat smooth function and its zero Taylor series")
        ax.legend()
        plt.show()

        print(f"f(A)                       = {math.exp(-1.0/A**2):.10g}")
        print("Every Taylor coefficient   = 0")
        print("Taylor approximation away from 0 = 0")
        show_note("Taylor representation requires the remainder to tend to zero. Infinite differentiability alone does not guarantee analyticity.", "#fff7ed", COLORS["orange"])


flat_button.on_click(run_flat_lab)
display(widgets.VBox([flat_A, flat_scale, flat_button, flat_output]))

## Cell 8 — Probability generating functions, moments, and independent sums

For an $\mathbb N_0$-valued random variable,

$$
G_X(s)=\mathbb E[s^X]=\sum_{n=0}^{\infty}p_ns^n.
$$

Coefficients recover the distribution, derivatives at $1-$ recover factorial moments, and independence gives

$$
G_{X_1+\cdots+X_m}(s)=\prod_{j=1}^{m}G_{X_j}(s).
$$

Select a standard law and compare the base distribution with a sum of independent copies.

In [ ]:
pgf_family = widgets.Dropdown(
    options=[
        ("Bernoulli(p)", "bernoulli"),
        ("Binomial(m,p)", "binomial"),
        ("Poisson(lambda)", "poisson"),
        ("Geometric(p) on {0,1,...}", "geometric"),
    ],
    value="poisson",
    description="Law:",
    layout=widgets.Layout(width="500px"),
)
pgf_parameter = widgets.FloatSlider(
    value=2.5, min=0.1, max=8.0, step=0.1, description="$\\lambda$:",
    continuous_update=False, layout=widgets.Layout(width="500px")
)
pgf_m = widgets.IntSlider(
    value=5, min=1, max=20, step=1, description="m:",
    continuous_update=False, layout=widgets.Layout(width="500px")
)
pgf_copies = widgets.IntSlider(
    value=2, min=1, max=5, step=1, description="Copies:",
    continuous_update=False, layout=widgets.Layout(width="500px")
)
pgf_s = widgets.FloatSlider(
    value=0.70, min=0.0, max=1.0, step=0.01, description="s:",
    continuous_update=False, layout=widgets.Layout(width="500px")
)
pgf_button = style_button(widgets.Button(description="Build generating fn"), "chart-bar")
pgf_output = widgets.Output()


def standard_pmf(family, parameter, m, cutoff=100):
    n = np.arange(cutoff + 1)
    if family == "bernoulli":
        p = parameter
        pmf = np.zeros(cutoff + 1)
        pmf[:2] = [1.0-p, p]
        mean, variance = p, p*(1.0-p)
    elif family == "binomial":
        p = parameter
        pmf = np.zeros(cutoff + 1)
        for k in range(m + 1):
            pmf[k] = math.comb(m, k) * p**k * (1.0-p)**(m-k)
        mean, variance = m*p, m*p*(1.0-p)
    elif family == "poisson":
        lam = parameter
        log_pmf = -lam + n*np.log(lam) - gammaln(n+1.0)
        pmf = np.exp(log_pmf)
        mean = variance = lam
    else:
        p = parameter
        pmf = p*(1.0-p)**n
        mean, variance = (1.0-p)/p, (1.0-p)/p**2
    return pmf, mean, variance


def run_pgf_lab(_=None):
    with pgf_output:
        clear_output(wait=True)
        family, parameter, m = pgf_family.value, pgf_parameter.value, pgf_m.value
        copies, s = pgf_copies.value, pgf_s.value
        base, mean, variance = standard_pmf(family, parameter, m)
        summed = np.array([1.0])
        for _ in range(copies):
            summed = np.convolve(summed, base)
        base_n = np.arange(len(base))
        sum_n = np.arange(len(summed))
        G_base = float(np.sum(base*s**base_n))
        G_sum = float(np.sum(summed*s**sum_n))

        base_limit = min(len(base), max(15, int(mean+6*math.sqrt(max(variance, 1e-9)))+1))
        sum_mean, sum_variance = copies*mean, copies*variance
        sum_limit = min(len(summed), max(20, int(sum_mean+6*math.sqrt(max(sum_variance, 1e-9)))+1))

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.1))
        axes[0].bar(base_n[:base_limit], base[:base_limit], color=COLORS["blue"], alpha=0.8)
        axes[0].set_title("Base probability mass function")
        axes[1].bar(sum_n[:sum_limit], summed[:sum_limit], color=COLORS["purple"], alpha=0.8)
        axes[1].set_title(rf"Sum of {copies} independent copies")
        plt.tight_layout()
        plt.show()

        print(f"Base mean and variance        = {mean:.8g}, {variance:.8g}")
        print(f"Sum mean and variance         = {sum_mean:.8g}, {sum_variance:.8g}")
        print(f"G_X(s) from coefficients      = {G_base:.12g}")
        print(f"G_sum(s) from convolution     = {G_sum:.12g}")
        print(f"[G_X(s)]^copies               = {G_base**copies:.12g}")
        print(f"Truncated base probability    = {np.sum(base):.12g}")
        show_note("Multiplication of PGFs is the analytic form of convolution for independent sums.")


def configure_pgf(change=None):
    family = pgf_family.value
    pgf_m.disabled = family != "binomial"
    if family == "poisson":
        pgf_parameter.description = "$\\lambda$:"
        pgf_parameter.min, pgf_parameter.max, pgf_parameter.step = 0.1, 8.0, 0.1
    else:
        pgf_parameter.description = "p:"
        pgf_parameter.min, pgf_parameter.max, pgf_parameter.step = 0.05, 0.95, 0.05
        if not (0.05 <= pgf_parameter.value <= 0.95):
            pgf_parameter.value = 0.40


pgf_family.observe(configure_pgf, names="value")
pgf_button.on_click(run_pgf_lab)
configure_pgf()
display(widgets.VBox([pgf_family, pgf_parameter, pgf_m, pgf_copies, pgf_s, pgf_button, pgf_output]))

## Cell 9 — Galton--Watson iteration and extinction probability

For binary reproduction with

$$
G(s)=1-p+ps^2,
$$

the extinction probabilities satisfy

$$
q_0=0,\qquad q_{n+1}=G(q_n).
$$

Their limit is the smallest fixed point of $G(s)=s$. The threshold is the mean offspring number $G'(1)=2p$.

In [ ]:
branch_p = widgets.FloatSlider(
    value=0.65, min=0.0, max=1.0, step=0.01, description="p:",
    continuous_update=False, layout=widgets.Layout(width="480px")
)
branch_generations = widgets.IntSlider(
    value=20, min=1, max=100, step=1, description="Generations:",
    continuous_update=False, layout=widgets.Layout(width="480px")
)
branch_button = style_button(widgets.Button(description="Iterate PGF"), "sitemap")
branch_output = widgets.Output()


def binary_extinction_sequence(p, generations):
    q = [0.0]
    for _ in range(generations):
        q.append(1.0-p+p*q[-1]**2)
    theoretical = 1.0 if p <= 0.5 or p == 0 else (1.0-p)/p
    return np.array(q), theoretical


def run_branching_lab(_=None):
    with branch_output:
        clear_output(wait=True)
        p, generations = branch_p.value, branch_generations.value
        q, theoretical = binary_extinction_sequence(p, generations)
        s = np.linspace(0.0, 1.0, 800)
        G = 1.0-p+p*s**2

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.1))
        axes[0].plot(s, G, color=COLORS["blue"], label="$G(s)$")
        axes[0].plot(s, s, color=COLORS["red"], ls="--", label="$s$")
        axes[0].scatter([theoretical], [theoretical], color=COLORS["green"], s=65, zorder=3)
        axes[0].set_title("Fixed points of the offspring PGF")
        axes[0].legend()
        axes[1].plot(np.arange(len(q)), q, marker="o", color=COLORS["purple"])
        axes[1].axhline(theoretical, color=COLORS["red"], ls="--", label="eventual extinction probability")
        axes[1].set_xlabel("generation $n$")
        axes[1].set_title(r"$q_n=\mathbb P(Z_n=0)$")
        axes[1].legend()
        plt.tight_layout()
        plt.show()

        print(f"Mean offspring number 2p = {2*p:.6g}")
        print(f"q_{generations}               = {q[-1]:.12g}")
        print(f"Theoretical extinction q    = {theoretical:.12g}")
        if p <= 0.5:
            show_note("The mean offspring number is at most one, so extinction is certain.")
        else:
            show_note("The smallest fixed point is below one, so survival has positive probability.")


branch_button.on_click(run_branching_lab)
display(widgets.VBox([branch_p, branch_generations, branch_button, branch_output]))

## Cell 10 — Bisection, Newton iteration, and a hybrid method

Bisection provides a certified bracket and the bound

$$
|x_n-x^*|\le\frac{b-a}{2^{n+1}}.
$$

Near a simple root, Newton's update

$$
x_{n+1}=x_n-\frac{f(x_n)}{f'(x_n)}
$$

converges quadratically. A hybrid method uses bisection for reliability and Newton for speed.

In [ ]:
root_family = widgets.Dropdown(
    options=[
        ("x^2-2 on [1,2]", "sqrt2"),
        ("x^3-2x-5 on [2,3]", "cubic"),
        ("cos(x)-x on [0,1]", "cosine"),
    ],
    value="cubic",
    description="Equation:",
    layout=widgets.Layout(width="500px"),
)
bisection_steps = widgets.IntSlider(
    value=4, min=0, max=20, step=1, description="Bisection steps:",
    continuous_update=False, layout=widgets.Layout(width="500px")
)
newton_steps = widgets.IntSlider(
    value=5, min=1, max=12, step=1, description="Newton steps:",
    continuous_update=False, layout=widgets.Layout(width="500px")
)
root_button = style_button(widgets.Button(description="Run hybrid method"), "crosshairs")
root_output = widgets.Output()


def root_problem(family):
    if family == "sqrt2":
        return (lambda x: x*x-2.0), (lambda x: 2.0*x), 1.0, 2.0
    if family == "cubic":
        return (lambda x: x**3-2.0*x-5.0), (lambda x: 3.0*x*x-2.0), 2.0, 3.0
    return (lambda x: math.cos(x)-x), (lambda x: -math.sin(x)-1.0), 0.0, 1.0


def bisection_history(f, a, b, steps):
    history = []
    left, right = float(a), float(b)
    for _ in range(steps + 1):
        midpoint = 0.5*(left+right)
        history.append((left, right, midpoint))
        if f(midpoint) == 0:
            left = right = midpoint
            break
        if f(left)*f(midpoint) <= 0:
            right = midpoint
        else:
            left = midpoint
    return history


def newton_history(f, df, x0, steps):
    values = [float(x0)]
    for _ in range(steps):
        derivative = df(values[-1])
        if abs(derivative) < 1e-14:
            break
        values.append(values[-1]-f(values[-1])/derivative)
    return np.array(values)


def run_root_lab(_=None):
    with root_output:
        clear_output(wait=True)
        f, df, a, b = root_problem(root_family.value)
        exact = brentq(f, a, b)
        bis = bisection_history(f, a, b, bisection_steps.value)
        start = bis[-1][2]
        newton = newton_history(f, df, start, newton_steps.value)
        bis_mid = np.array([item[2] for item in bis])
        bis_error = np.abs(bis_mid-exact)
        newton_error = np.abs(newton-exact)
        bounds = np.array([(item[1]-item[0])/2.0 for item in bis])

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.1))
        axes[0].semilogy(np.arange(len(bis_error)), np.maximum(bis_error,1e-18), "o-", color=COLORS["blue"], label="actual bisection error")
        axes[0].semilogy(np.arange(len(bounds)), np.maximum(bounds,1e-18), "--", color=COLORS["red"], label="certified bound")
        axes[0].set_title("Reliable linear convergence")
        axes[0].legend()
        axes[1].semilogy(np.arange(len(newton_error)), np.maximum(newton_error,1e-18), "o-", color=COLORS["purple"])
        axes[1].set_title("Local Newton convergence")
        axes[1].set_xlabel("Newton step")
        plt.tight_layout()
        plt.show()

        print(f"Reference root           = {exact:.15g}")
        print(f"Hybrid starting value    = {start:.15g}")
        print("Newton iterates:")
        for index, value in enumerate(newton):
            print(f"  {index:2d}: {value:.15g}   error={abs(value-exact):.3e}")
        show_note("The bracket certifies existence and location; the Newton phase exploits the simple-root geometry for rapid local convergence.")


root_button.on_click(run_root_lab)
display(widgets.VBox([root_family, bisection_steps, newton_steps, root_button, root_output]))

## Cell 11 — Composite trapezoidal rule with a certified error

For $h=(b-a)/n$,

$$
T_n=h\left(\frac{f(a)}2+\sum_{k=1}^{n-1}f(a+kh)+\frac{f(b)}2\right),
$$

and

$$
\left|\int_a^bf-T_n\right|
\le\frac{b-a}{12}h^2\|f''\|_\infty.
$$

The observed error may be much smaller, but the theorem bound is guaranteed before the exact integral is known.

In [ ]:
trap_family = widgets.Dropdown(
    options=[
        ("exp(-x^2) on [0,1]", "gaussian"),
        ("exp(x) on [0,1]", "exp"),
        ("sin(x) on [0,pi]", "sin"),
        ("x^4 on [0,1]", "quartic"),
    ],
    value="gaussian",
    description="Integrand:",
    layout=widgets.Layout(width="480px"),
)
trap_panels = widgets.IntSlider(
    value=12, min=1, max=200, step=1, description="Panels n:",
    continuous_update=False, layout=widgets.Layout(width="480px")
)
trap_button = style_button(widgets.Button(description="Apply trapezoids"), "draw-polygon")
trap_output = widgets.Output()


def trapezoid_problem(family):
    if family == "gaussian":
        return (lambda x: np.exp(-x*x)), 0.0, 1.0, 2.0
    if family == "exp":
        return np.exp, 0.0, 1.0, math.e
    if family == "sin":
        return np.sin, 0.0, math.pi, 1.0
    return (lambda x: x**4), 0.0, 1.0, 12.0


def composite_trapezoid(f, a, b, panels):
    x = np.linspace(a, b, panels+1)
    values = f(x)
    h = (b-a)/panels
    return h*(0.5*values[0]+np.sum(values[1:-1])+0.5*values[-1])


def run_trapezoid_lab(_=None):
    with trap_output:
        clear_output(wait=True)
        f, a, b, second_bound = trapezoid_problem(trap_family.value)
        panels = trap_panels.value
        approximation = float(composite_trapezoid(f,a,b,panels))
        exact = quad(lambda t: float(f(t)),a,b)[0]
        h = (b-a)/panels
        bound = (b-a)*h*h*second_bound/12.0

        dense = np.linspace(a,b,1601)
        nodes = np.linspace(a,b,panels+1)
        fig, axes = plt.subplots(1, 2, figsize=(12,4.1))
        axes[0].plot(dense,f(dense),color=COLORS["blue"],lw=2,label="$f$")
        axes[0].plot(nodes,f(nodes),color=COLORS["orange"],marker="o",label="piecewise linear")
        axes[0].fill_between(nodes,0,f(nodes),color=COLORS["orange"],alpha=0.15)
        axes[0].set_title("Composite trapezoidal approximation")
        axes[0].legend()

        panel_grid = np.unique(np.geomspace(1,max(2,panels),80).astype(int))
        actual_errors = np.array([abs(composite_trapezoid(f,a,b,int(k))-exact) for k in panel_grid])
        theorem_bounds = (b-a)**3*second_bound/(12.0*panel_grid**2)
        axes[1].loglog(panel_grid,actual_errors,"o-",ms=3,color=COLORS["purple"],label="actual error")
        axes[1].loglog(panel_grid,theorem_bounds,"--",color=COLORS["red"],label="certified bound")
        axes[1].set_title("Quadratic error decay")
        axes[1].legend()
        plt.tight_layout()
        plt.show()

        print(f"Trapezoidal approximation = {approximation:.15g}")
        print(f"Reference integral         = {exact:.15g}")
        print(f"Observed error             = {abs(approximation-exact):.6e}")
        print(f"Certified error bound      = {bound:.6e}")
        show_note("Doubling the panel count divides the theoretical error bound by four.")


trap_button.on_click(run_trapezoid_lab)
display(widgets.VBox([trap_family, trap_panels, trap_button, trap_output]))

## Cell 12 — Infinite products and logarithmic series

For positive factors,

$$
\prod_{n=1}^{\infty}a_n\text{ converges to }P>0
\quad\Longleftrightarrow\quad
\sum_{n=1}^{\infty}\log a_n\text{ converges},
$$

and then $P=\exp(\sum\log a_n)$. For factors $1+u_n$, the second-order term

$$
\log(1+u_n)=u_n-\frac12u_n^2+O(u_n^3)
$$

explains why conditional signed cases require more than convergence of $\sum u_n$.

In [ ]:
product_family = widgets.Dropdown(
    options=[
        ("1+1/n: product grows like N+1", "harmonic_positive"),
        ("1+1/n^2: convergent positive product", "square_positive"),
        ("1-1/(n+1): product tends to zero", "telescoping_zero"),
        ("1+(-1)^n/n, n>=2: conditional convergence", "alternating"),
        ("1+(-1)^n/sqrt(n), n>=2: product tends to zero", "quadratic_obstruction"),
    ],
    value="square_positive",
    description="Factors:",
    layout=widgets.Layout(width="590px"),
)
product_N = widgets.IntSlider(
    value=500, min=10, max=5000, step=10, description="N:",
    continuous_update=False, layout=widgets.Layout(width="590px")
)
product_button = style_button(widgets.Button(description="Multiply safely"), "times")
product_output = widgets.Output()


def product_perturbations(family, N):
    start = 2 if family in {"alternating","quadratic_obstruction"} else 1
    n = np.arange(start,N+1,dtype=float)
    if family == "harmonic_positive":
        u = 1.0/n
    elif family == "square_positive":
        u = 1.0/n**2
    elif family == "telescoping_zero":
        u = -1.0/(n+1.0)
    elif family == "alternating":
        u = (-1.0)**n.astype(int)/n
    else:
        u = (-1.0)**n.astype(int)/np.sqrt(n)
    return n,u


def run_product_lab(_=None):
    with product_output:
        clear_output(wait=True)
        family,N=product_family.value,product_N.value
        n,u=product_perturbations(family,N)
        log_partials=np.cumsum(np.log1p(u))
        products=np.exp(log_partials)
        sum_u=np.cumsum(u)
        sum_u2=np.cumsum(u*u)

        fig,axes=plt.subplots(1,2,figsize=(12,4.1))
        axes[0].plot(n,products,color=COLORS["blue"],lw=2)
        axes[0].set_title("Partial products computed through logarithms")
        axes[0].set_xlabel("$N$")
        axes[1].plot(n,sum_u,label=r"$\sum u_n$",color=COLORS["purple"])
        axes[1].plot(n,sum_u2,label=r"$\sum u_n^2$",color=COLORS["orange"])
        axes[1].set_title("First- and second-order diagnostics")
        axes[1].legend()
        plt.tight_layout()
        plt.show()

        print(f"Final partial product      = {products[-1]:.12g}")
        print(f"Final logarithmic sum      = {log_partials[-1]:.12g}")
        print(f"Final sum of u_n           = {sum_u[-1]:.12g}")
        print(f"Final sum of u_n^2         = {sum_u2[-1]:.12g}")
        messages={
            "harmonic_positive":"The factors tend to one, but the product equals $N+1$ and diverges to infinity.",
            "square_positive":"Nonnegative perturbations with a convergent $\\sum1/n^2$ produce a positive convergent product.",
            "telescoping_zero":"The exact partial product is $1/(N+1)$, so the product diverges to zero under the chapter's definition.",
            "alternating":"Both $\\sum u_n$ and $\\sum u_n^2$ converge; the second-order criterion gives a positive product limit.",
            "quadratic_obstruction":"Although $\\sum u_n$ converges, $\\sum u_n^2$ is harmonic. The negative quadratic logarithmic correction drives the product to zero.",
        }
        show_note(messages[family],"#fff7ed" if family in {"harmonic_positive","telescoping_zero","quadratic_obstruction"} else "#eff6ff",COLORS["orange"] if family in {"harmonic_positive","telescoping_zero","quadratic_obstruction"} else COLORS["blue"])


product_button.on_click(run_product_lab)
display(widgets.VBox([product_family,product_N,product_button,product_output]))

## Cell 13 — Interactive theorem and method selector

A rigorous solution should connect

$$
\text{visible structure}\Longrightarrow
\text{verified hypotheses}\Longrightarrow
\text{named theorem or algorithm}.
$$

Choose the problem structure below to review the decisive check and the main warning.

In [ ]:
method_cases={
    "A power series with known coefficients":(
        "Cauchy--Hadamard or ratio formula",
        r"Compute $\limsup|a_n|^{1/n}$ or the coefficient ratio, then test both finite endpoints separately.",
        "The radius gives no automatic endpoint conclusion."
    ),
    "Differentiating or integrating a power series":(
        "Compact uniform convergence inside the radius",
        "Work on a closed interval strictly inside the radius; the derived and integrated series have the same radius.",
        "Boundary operations require separate justification."
    ),
    "Approaching a convergent endpoint":(
        "Abel's boundary theorem",
        r"Verify convergence of the endpoint coefficient series $\sum a_n$.",
        "The theorem is one-sided and does not assert convergence of a divergent endpoint series."
    ),
    "A finite Taylor approximation":(
        "Taylor theorem with a remainder",
        r"Bound the appropriate derivative on the interval joining $x_0$ and $x$.",
        "A small observed error is not a certified remainder bound."
    ),
    "An infinitely differentiable function":(
        "Taylor remainder criterion",
        r"Prove $R_n(f;x_0)(x)\to0$; smoothness alone is insufficient.",
        "The flat function has every Taylor coefficient zero but is not locally zero."
    ),
    "An independent sum of integer-valued variables":(
        "Probability generating functions",
        r"Multiply the PGFs and recover coefficients or moments by differentiation.",
        "Independence is required for the product identity."
    ),
    "A branching-process extinction question":(
        "Iterated PGF and smallest fixed point",
        r"Iterate $q_{n+1}=G(q_n)$ from $q_0=0$ and solve $G(q)=q$ in $[0,1]$.",
        "Choose the smallest fixed point, not automatically $q=1$."
    ),
    "A continuous function changes sign":(
        "Bisection",
        r"Maintain a sign-changing bracket and use the interval-length error bound.",
        "Bisection is reliable but only linearly convergent."
    ),
    "A good initial guess near a simple root":(
        "Newton's method",
        r"Check $f'(x^*)\ne0$ and that the iteration remains in a valid local neighbourhood.",
        "A poor starting value or a zero derivative can destroy the iteration."
    ),
    "A twice differentiable integrand":(
        "Composite trapezoidal rule",
        r"Bound $\|f''\|_\infty$ and use $(b-a)h^2\|f''\|_\infty/12$.",
        "The theorem bound is generally larger than the observed error."
    ),
    "A positive infinite product":(
        "Logarithmic product criterion",
        r"Study $\sum\log a_n$; for $a_n=1+u_n$, inspect the quadratic correction.",
        r"The necessary condition $a_n\to1$ is not sufficient."
    ),
}

method_selector=widgets.Dropdown(options=list(method_cases.keys()),description="Structure:",layout=widgets.Layout(width="620px"))
method_button=style_button(widgets.Button(description="Suggest method"),"compass")
method_output=widgets.Output()


def suggest_method(_=None):
    with method_output:
        clear_output(wait=True)
        method,check,warning=method_cases[method_selector.value]
        display(Markdown(f"### Suggested method: **{method}**"))
        display(Markdown(f"**Decisive check.** {check}"))
        display(Markdown(f"**Warning.** {warning}"))


method_button.on_click(suggest_method)
display(widgets.VBox([method_selector,method_button,method_output]))
suggest_method()

## Cell 14 — Self-check quiz with immediate feedback

The chapter repeatedly separates an interior theorem from its boundary or error hypotheses. Answer one question at a time and justify the implication before pressing **Check answer**.

In [ ]:
quiz_questions=[
    {
        "prompt":r"What does the radius theorem say at $|x-x_0|=R$?",
        "options":["Absolute convergence","Divergence","No general conclusion"],
        "answer":"No general conclusion",
        "explanation":"Each endpoint must be tested as a separate numerical series."
    },
    {
        "prompt":r"Where may a power series be differentiated term by term automatically?",
        "options":["Strictly inside its radius","At every endpoint","Outside its radius"],
        "answer":"Strictly inside its radius",
        "explanation":"The derivative series has the same radius and converges uniformly on smaller closed intervals."
    },
    {
        "prompt":r"What must hold for a smooth function to equal its Taylor series at $x$?",
        "options":[r"$f'(x)=0$",r"The Taylor remainder tends to zero",r"All derivatives are bounded at one point"],
        "answer":r"The Taylor remainder tends to zero",
        "explanation":"Infinite differentiability alone is not enough."
    },
    {
        "prompt":r"For a PGF, what is $G_X'(1-)$ when finite?",
        "options":[r"$\mathbb P(X=0)$",r"$\mathbb E[X]$",r"$\operatorname{Var}(X)$"],
        "answer":r"$\mathbb E[X]$",
        "explanation":"The second derivative gives the second factorial moment, which combines with the first to produce the variance."
    },
    {
        "prompt":"Which root of the branching fixed-point equation gives extinction probability?",
        "options":["The largest real root","The smallest root in [0,1]","Always the root 1"],
        "answer":"The smallest root in [0,1]",
        "explanation":"Iteration from q_0=0 increases to the smallest fixed point."
    },
    {
        "prompt":"What is the main advantage of bisection over Newton iteration?",
        "options":["Quadratic speed","A certified bracket under continuity and a sign change","No function evaluations"],
        "answer":"A certified bracket under continuity and a sign change",
        "explanation":"Newton is faster locally, but bisection provides global reliability under weaker information."
    },
    {
        "prompt":r"For a positive infinite product, what series gives an exact convergence criterion?",
        "options":[r"$\sum a_n$",r"$\sum\log a_n$",r"$\sum a_n^2$"],
        "answer":r"$\sum\log a_n$",
        "explanation":r"The logarithm converts partial products into partial sums."
    },
]

quiz_state={"index":0,"score":0,"answered":False}
quiz_prompt=widgets.HTMLMath()
quiz_choice=widgets.ToggleButtons(layout=widgets.Layout(width="100%"))
quiz_check=style_button(widgets.Button(description="Check answer"),"check")
quiz_next=widgets.Button(description="Next question",icon="arrow-right",disabled=True)
quiz_restart=widgets.Button(description="Restart quiz",icon="redo")
quiz_feedback=widgets.Output()
quiz_score=widgets.HTML()


def render_quiz_question():
    q=quiz_questions[quiz_state["index"]]
    quiz_prompt.value=f"<h4>Question {quiz_state['index']+1} of {len(quiz_questions)}</h4><p>{q['prompt']}</p>"
    quiz_choice.options=q["options"]
    quiz_choice.value=None
    quiz_state["answered"]=False
    quiz_check.disabled=False
    quiz_next.disabled=True
    quiz_score.value=f"<b>Score:</b> {quiz_state['score']} / {quiz_state['index']} completed"
    with quiz_feedback:
        clear_output()


def check_quiz_answer(_=None):
    if quiz_state["answered"]:
        return
    with quiz_feedback:
        clear_output(wait=True)
        if quiz_choice.value is None:
            display(Markdown("Please choose an answer first."))
            return
        q=quiz_questions[quiz_state["index"]]
        correct=quiz_choice.value==q["answer"]
        if correct:
            quiz_state["score"]+=1
            display(Markdown("### Correct"))
        else:
            display(Markdown(f"### Not yet — the correct answer is **{q['answer']}**"))
        display(Markdown(q["explanation"]))
    quiz_state["answered"]=True
    quiz_check.disabled=True
    quiz_next.disabled=False
    quiz_score.value=f"<b>Score:</b> {quiz_state['score']} / {quiz_state['index']+1} completed"


def next_quiz_question(_=None):
    if quiz_state["index"]<len(quiz_questions)-1:
        quiz_state["index"]+=1
        render_quiz_question()
    else:
        with quiz_feedback:
            clear_output(wait=True)
            display(Markdown(f"## Quiz complete: {quiz_state['score']} / {len(quiz_questions)}"))
        quiz_next.disabled=True


def restart_quiz(_=None):
    quiz_state.update(index=0,score=0,answered=False)
    render_quiz_question()


quiz_check.on_click(check_quiz_answer)
quiz_next.on_click(next_quiz_question)
quiz_restart.on_click(restart_quiz)
render_quiz_question()
display(widgets.VBox([quiz_prompt,quiz_choice,widgets.HBox([quiz_check,quiz_next,quiz_restart]),quiz_score,quiz_feedback]))

## Cell 15 — Practice generator: theorem, hypotheses, computation

For each problem, first identify the relevant radius, remainder, generating function, error bound, or logarithmic series. Then verify every hypothesis before calculating.

In [ ]:
practice_bank=[
    {
        "problem":r"Find the interval of convergence of $\sum_{n=0}^{\infty}x^n/(n+1)$.",
        "hint":"The radius is one; substitute x=1 and x=-1 separately.",
        "solution":r"At $x=1$ the harmonic series diverges, while at $x=-1$ the alternating series converges. The interval is $[-1,1)$."
    },
    {
        "problem":r"Use the geometric series to evaluate $\sum_{n=1}^{\infty}n/2^n$.",
        "hint":r"Differentiate $\sum x^n=(1-x)^{-1}$ and multiply by x.",
        "solution":r"For $|x|<1$, $\sum nx^n=x/(1-x)^2$. At $x=1/2$ the sum is $2$."
    },
    {
        "problem":r"Find a certified degree ensuring $|e-T_n(e^x;0)(1)|<10^{-6}$.",
        "hint":r"Use $|R_n(1)|\le e/(n+1)!$ and increase n until the bound is small enough.",
        "solution":r"The first sufficient choice is $n=9$, since $e/10!<10^{-6}$, while $e/9!>10^{-6}$."
    },
    {
        "problem":r"Recover the mean and variance from $G(s)=e^{\lambda(s-1)}$.",
        "hint":r"Evaluate $G'(1)$ and $G''(1)+G'(1)-G'(1)^2$.",
        "solution":r"One obtains $G'(1)=\lambda$ and variance $\lambda^2+\lambda-\lambda^2=\lambda$."
    },
    {
        "problem":"For binary reproduction with p=0.7, find the eventual extinction probability.",
        "hint":r"Solve $0.7s^2-s+0.3=0$ and select the smallest root in $[0,1]$.",
        "solution":r"The roots are $1$ and $3/7$. The extinction probability is $q=3/7$."
    },
    {
        "problem":r"How many bisection reductions guarantee error at most $10^{-6}$ on an initial interval of length one?",
        "hint":r"Require $2^{-(n+1)}\le10^{-6}$.",
        "solution":r"It is enough to take $n\ge\lceil\log_2(10^6)\rceil-1=19$ reductions."
    },
    {
        "problem":r"Choose panels so that the trapezoidal error for $\int_0^1e^{-x^2}dx$ is at most $10^{-4}$.",
        "hint":r"Use the bound $1/(6n^2)$.",
        "solution":r"Require $1/(6n^2)\le10^{-4}$, giving $n\ge\sqrt{10000/6}$. Thus $n=41$ is sufficient."
    },
    {
        "problem":r"Classify $\prod_{n=1}^{\infty}(1+1/n^2)$.",
        "hint":r"The perturbations are nonnegative and $\sum1/n^2$ converges.",
        "solution":r"The product converges to a finite positive limit by the nonnegative-perturbation criterion."
    },
]

practice_rng=random.Random(2026)
practice_state={"item":None}
practice_output=widgets.Output()
new_problem_button=style_button(widgets.Button(description="New problem"),"dice")
hint_button=widgets.Button(description="Show hint",icon="lightbulb")
solution_button=widgets.Button(description="Reveal solution",icon="eye")


def new_practice_problem(_=None):
    practice_state["item"]=practice_rng.choice(practice_bank)
    with practice_output:
        clear_output(wait=True)
        display(Markdown("### Problem"))
        display(Markdown(practice_state["item"]["problem"]))
        display(Markdown("_Name the theorem and verify its hypotheses before computing._"))


def show_practice_hint(_=None):
    if practice_state["item"] is None:
        new_practice_problem()
    with practice_output:
        display(Markdown(f"**Hint.** {practice_state['item']['hint']}"))


def show_practice_solution(_=None):
    if practice_state["item"] is None:
        new_practice_problem()
    with practice_output:
        display(Markdown(f"**Solution.** {practice_state['item']['solution']}"))


new_problem_button.on_click(new_practice_problem)
hint_button.on_click(show_practice_hint)
solution_button.on_click(show_practice_solution)
display(widgets.VBox([widgets.HBox([new_problem_button,hint_button,solution_button]),practice_output]))
new_practice_problem()

## Suggested learning path and extensions

1. In the radius laboratory, approach both endpoints from inside and compare the term test with the partial sums.
2. Increase $r$ toward $1$ in the geometric-operation laboratory and explain why compact uniform bounds deteriorate.
3. Compare the alternating and positive harmonic families in the Abel laboratory.
4. For $e^x$, $\sin x$, and $\log(1+x)$, compare the observed Taylor error with the certified bound.
5. Use the fast series for $\log2$ and compare its error with the alternating harmonic partial sum using the same number of terms.
6. In the PGF laboratory, verify numerically that sums of Poisson variables remain Poisson by comparing probabilities.
7. Move the binary branching parameter through the threshold $p=1/2$.
8. Compare bisection and Newton errors on a logarithmic scale and identify the change from linear to quadratic convergence.
9. Double the trapezoidal panel count and verify the factor-of-four improvement in the theorem bound.
10. Compare the two alternating infinite products and explain the role of $\sum u_n^2$.

### Final reminder

Computation supports the theory, but the conclusion depends on the correct domain, endpoint test, remainder estimate, independence assumption, simple-root condition, or logarithmic criterion.